In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_contents


In [ ]:
# Constants
MODEL_OLLAMA = "gpt-oss:20b"
MODEL_GEMINI = "gemma3:270m"

In [ ]:
ollama = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1"
)
gemini = OpenAI(
    api_key="gemini",
    base_url="http://127.0.0.1:11434/v1"
)


In [ ]:

def stream_ollama(prompt):
    messages = [
        {
            "role": "system", "content": system_message,
        },
        {
            "role": "user", "content": prompt
        }
    ]
    stream = ollama.chat.completions.create(
        model=MODEL_OLLAMA,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ''
        yield result



In [ ]:
# Test stream_ollama
message_input = gr.Textbox(
    label="Your message:", info="Enter a message for Ollama", lines=7
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_ollama,
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
    title="Ollama"
)
view.launch(inbrowser=True)

In [ ]:
def stream_gemini(prompt):
    messages = [
        {
            "role": "system", "content": system_message,
        },
        {
            "role": "user", "content": prompt,
        }
    ]

    stream = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [ ]:
# Test stream_gemini
message_input = gr.Textbox(
    label="Your message:", info="Enter a message for Gemini", lines=7
)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gemini,
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
    title="Gemini"
)
view.launch(inbrowser=True)

In [ ]:
def stream_model(prompt, model):
    if model == "Ollama":
        result = stream_ollama(prompt)
    elif model == "Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")

    yield from result

In [ ]:
message_input = gr.Textbox(
    label="Your message:", info="Enter a message for the LLM", lines=7
)
model_selector = gr.Dropdown(["Ollama", "Gemini"], label="Select model", value="Ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", "Ollama"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Gemini"],
    ],
    flagging_mode="never",
    title="LLMs",
)
view.launch(inbrowser=True)

## Finallly the stream brochure implementation

In [ ]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""


def stream_brochure(company_name, url, model):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model == "Ollama":
       result = stream_ollama(prompt)
    elif model == "gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown Model")

    yield from result


In [ ]:
company_name_input = gr.Textbox(label="Company Name")
company_url_input = gr.Textbox(label="Landing Page URL including http:// or https://")
model_selector = gr.Dropdown(["Ollama", "Gemini"], label="Select Model", value="Ollama")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[company_name_input, company_url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "Ollama"],
        ["Edward Donner", "https://edwarddonner.com", "Gemini"]
    ],
    flagging_mode="never"
    )
view.launch()
